# Week 5: Can a Neural Network Predict Loan Approval?

[Open in Colab](https://colab.research.google.com/github/astarryknight/ai-ml/blob/main/notebooks/week-05-neural-networks.ipynb)

In this notebook, you will build a small **fully connected neural network** that predicts whether a loan application was approved using six understandable financial inputs. You will see the finished project first, learn the theory behind it, and then rebuild it one step at a time.

> **Important:** This is an educational model that learns labels in a Kaggle dataset. It must not be used to decide whether a real person receives credit. Real lending decisions require verified data, legal compliance, fairness testing, explanations, human review, and an appeals process.

## First: see the finished project

Run the next cell before worrying about every line. It downloads the dataset, trains a neural network, and tests it on loan applications the model did not see during training.

Look for three things:

1. How much the network improves over always guessing the most common outcome.
2. How many approved and rejected applications it classifies correctly.
3. How the predicted approval probability changes for example applicants.

In [ ]:
# FINAL DEMO — run this now; we will rebuild it below.
import subprocess
import sys

try:
    import kagglehub
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
    import kagglehub

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

demo_path = kagglehub.dataset_download(
    "architsharma01/loan-approval-prediction-dataset",
    path="loan_approval_dataset.csv",
)
demo_data = pd.read_csv(demo_path)
demo_data.columns = demo_data.columns.str.strip()
demo_data["loan_status"] = (
    demo_data["loan_status"].str.strip().map({"Approved": 1, "Rejected": 0})
)

demo_features = [
    "income_annum", "loan_amount", "loan_term",
    "cibil_score", "residential_assets_value", "bank_asset_value",
]
X_demo_train, X_demo_test, y_demo_train, y_demo_test = train_test_split(
    demo_data[demo_features], demo_data["loan_status"],
    test_size=0.20, random_state=42, stratify=demo_data["loan_status"],
)

demo_scaler = StandardScaler()
X_demo_train_scaled = demo_scaler.fit_transform(X_demo_train)
X_demo_test_scaled = demo_scaler.transform(X_demo_test)

demo_network = MLPClassifier(
    hidden_layer_sizes=(8,), activation="relu", solver="adam",
    learning_rate_init=0.003, alpha=0.1, max_iter=1000,
    early_stopping=True, validation_fraction=0.15,
    n_iter_no_change=50, batch_size=128, random_state=42,
)
demo_network.fit(X_demo_train_scaled, y_demo_train)

demo_probabilities = demo_network.predict_proba(X_demo_test_scaled)[:, 1]
demo_predictions = (demo_probabilities >= 0.50).astype(int)
demo_baseline = np.repeat(y_demo_train.mode().iloc[0], len(y_demo_test))

print(f"Majority-class baseline accuracy: {accuracy_score(y_demo_test, demo_baseline):.1%}")
print(f"Neural-network accuracy:        {accuracy_score(y_demo_test, demo_predictions):.1%}")
print(f"Balanced accuracy:              {balanced_accuracy_score(y_demo_test, demo_predictions):.1%}")
print(f"Precision for approvals:        {precision_score(y_demo_test, demo_predictions):.1%}")
print(f"Recall for approvals:           {recall_score(y_demo_test, demo_predictions):.1%}")
print(f"ROC-AUC:                        {roc_auc_score(y_demo_test, demo_probabilities):.1%}")

ConfusionMatrixDisplay.from_predictions(
    y_demo_test, demo_predictions, display_labels=["Rejected", "Approved"],
    cmap="Blues", colorbar=False,
)
plt.title("Finished project: held-out loan applications")
plt.tight_layout()
plt.show()

## What just happened?

The project followed a familiar machine-learning workflow:

**income + requested amount + term + credit score + assets → neural network → approval probability**

The baseline always chooses the most common result. The neural network instead learns patterns across six financial inputs and performs strongly on a held-out test set. High performance here means the dataset contains a consistent approval pattern—it does **not** prove the model is fair or suitable for real lending.

## Theory in five minutes

### A fully connected network

Our network has six inputs, one hidden layer with eight neurons, and one output probability. Every input connects to every hidden neuron. Each neuron combines the inputs using learned weights and a bias.

**weighted sum = input 1 × weight 1 + input 2 × weight 2 + ... + bias**

The hidden neurons use **ReLU**, which keeps positive values and turns negative values into zero. The output becomes a number from 0 to 1: the model's estimated probability that the dataset label is `Approved`.

### How the network learns

1. The network produces an approval probability.
2. A loss function measures how far that probability is from the known label.
3. Backpropagation assigns part of the error to each weight.
4. The optimizer adjusts the weights and repeats.

The library handles those calculations. Our job is to select understandable inputs, hold back test examples, and evaluate more than one metric.

## Code deep dive

Now rebuild the demo in small pieces. Run the cells in order.

### 1. Import the tools

`MLPClassifier` is scikit-learn's fully connected neural network for classification. `kagglehub` downloads the public dataset directly from Kaggle.

In [ ]:
import subprocess
import sys

try:
    import kagglehub
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
    import kagglehub

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

### 2. Download the Kaggle dataset

The [Loan Approval Prediction Dataset](https://www.kaggle.com/datasets/architsharma01/loan-approval-prediction-dataset) contains 4,269 labeled applications. Its original collection process and currency units are not documented clearly, so we will call the monetary values **dataset units** and treat the data as an educational case study.

In [ ]:
csv_path = kagglehub.dataset_download(
    "architsharma01/loan-approval-prediction-dataset",
    path="loan_approval_dataset.csv",
)
loans = pd.read_csv(csv_path)
loans.columns = loans.columns.str.strip()

print(f"Rows: {len(loans):,}")
loans.head()

### 3. Choose six financial inputs

We intentionally leave out applicant ID, education, employment type, and number of dependents. The model receives only six financial values:

- Annual income
- Requested loan amount
- Loan term
- CIBIL credit score
- Residential asset value
- Bank asset value

The target text becomes a number: `Rejected = 0` and `Approved = 1`.

In [ ]:
feature_names = [
    "income_annum",
    "loan_amount",
    "loan_term",
    "cibil_score",
    "residential_assets_value",
    "bank_asset_value",
]

loans["loan_status"] = (
    loans["loan_status"].str.strip().map({"Rejected": 0, "Approved": 1})
)
data = loans[[*feature_names, "loan_status"]].copy()
data.head()

### 4. Inspect the labels

Before training, count both outcomes. Accuracy can be misleading when one class is much more common, so we will also calculate balanced accuracy, precision, recall, and ROC-AUC.

In [ ]:
label_counts = data["loan_status"].value_counts().sort_index()
label_counts.index = ["Rejected", "Approved"]
display(label_counts.to_frame("applications"))

label_counts.plot.bar(color=["#E07A5F", "#147D88"], rot=0)
plt.title("Outcomes in the complete dataset")
plt.ylabel("Applications")
plt.tight_layout()
plt.show()

### 5. Hold back a test set

The network trains on 80% of the applications. The other 20% stay hidden until evaluation. `stratify` keeps approximately the same approval proportion in both groups.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data[feature_names], data["loan_status"],
    test_size=0.20, random_state=42, stratify=data["loan_status"],
)

print("Training examples:", len(X_train))
print("Testing examples: ", len(X_test))

### 6. Scale using training data only

Income, assets, years, and credit scores use very different numeric ranges. Standardization puts them on comparable scales, which helps the network learn.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Inputs per application:", X_train_scaled.shape[1])

### 7. Establish the baseline

Our baseline always predicts the most common training label. A useful network must outperform that guess for both classes—not only overall.

In [ ]:
baseline_predictions = np.repeat(y_train.mode().iloc[0], len(y_test))
baseline_accuracy = accuracy_score(y_test, baseline_predictions)
baseline_balanced_accuracy = balanced_accuracy_score(y_test, baseline_predictions)

print(f"Baseline accuracy:          {baseline_accuracy:.1%}")
print(f"Baseline balanced accuracy: {baseline_balanced_accuracy:.1%}")

### 8. Train one small neural network

`hidden_layer_sizes=(8,)` creates one hidden layer with only eight neurons. Early stopping ends training when validation performance stops improving.

In [ ]:
network = MLPClassifier(
    hidden_layer_sizes=(8,),
    activation="relu",
    solver="adam",
    learning_rate_init=0.003,
    alpha=0.1,
    max_iter=1000,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=50,
    batch_size=128,
    random_state=42,
)

network.fit(X_train_scaled, y_train)
print("Training epochs:", network.n_iter_)

### 9. Turn probabilities into predictions

The model first outputs an approval probability. A threshold of 0.50 converts that probability into `Rejected` or `Approved`.

In [ ]:
approval_probabilities = network.predict_proba(X_test_scaled)[:, 1]
predictions = (approval_probabilities >= 0.50).astype(int)

accuracy = accuracy_score(y_test, predictions)
balanced_accuracy = balanced_accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)
auc = roc_auc_score(y_test, approval_probabilities)

print(f"Baseline accuracy:        {baseline_accuracy:.1%}")
print(f"Neural-network accuracy:  {accuracy:.1%}")
print(f"Balanced accuracy:        {balanced_accuracy:.1%}")
print(f"Precision for approvals:  {precision:.1%}")
print(f"Recall for approvals:     {recall:.1%}")
print(f"ROC-AUC:                  {auc:.1%}")

### 10. Read the confusion matrix

The diagonal boxes are correct predictions. The off-diagonal boxes are mistakes. This view prevents one impressive percentage from hiding which class the model misses.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, predictions,
    display_labels=["Rejected", "Approved"],
    cmap="Blues", colorbar=False,
)
plt.title("Predictions on held-out applications")
plt.tight_layout()
plt.show()

### 11. Make the model tangible

These example applicants use the dataset's undocumented monetary units. Edit the values and rerun the cell to see how the model's probability changes. The output describes the model—not who should receive a loan.

In [ ]:
applicants = pd.DataFrame({
    "income_annum": [4_100_000, 9_600_000, 6_500_000],
    "loan_amount": [12_200_000, 29_900_000, 18_000_000],
    "loan_term": [8, 12, 10],
    "cibil_score": [417, 778, 550],
    "residential_assets_value": [2_700_000, 2_400_000, 8_000_000],
    "bank_asset_value": [3_300_000, 8_000_000, 6_000_000],
}, index=["Applicant A", "Applicant B", "Applicant C"])

applicant_probabilities = network.predict_proba(
    scaler.transform(applicants)
)[:, 1]

applicant_results = applicants.copy()
applicant_results["approval_probability"] = applicant_probabilities
applicant_results["model_prediction"] = np.where(
    applicant_probabilities >= 0.50, "Approved", "Rejected"
)
applicant_display = applicant_results[[
    "cibil_score", "loan_term", "approval_probability", "model_prediction"
]].copy()
applicant_display["approval_probability"] = applicant_display["approval_probability"].map(
    lambda value: f"{value:.1%}"
)
applicant_display

### 12. Inspect the mistakes

Strong overall performance does not mean every prediction is correct. Look at the model's most confident mistakes.

In [ ]:
results = X_test.copy()
results["actual"] = y_test
results["predicted"] = predictions
results["approval_probability"] = approval_probabilities
results["confidence"] = np.where(
    predictions == 1, approval_probabilities, 1 - approval_probabilities
)

mistakes = results[results["actual"] != results["predicted"]]
mistakes.sort_values("confidence", ascending=False).head(8)

## Your experiment

Change one thing at a time and rerun the relevant cells:

- Edit one applicant's credit score or requested loan amount.
- Try 4 or 16 hidden neurons.
- Remove one input and see whether test performance changes.

Record accuracy, balanced accuracy, and ROC-AUC. Then explain why you think the result changed.

## What this model cannot tell us

This model predicts a label in one educational dataset; it does not determine whether an applicant can repay safely or whether the original decisions were fair. The dataset has limited provenance, and a highly predictable historical process can still be a harmful one. Real credit systems require documented data, bias and subgroup testing, explanations, privacy protections, legal review, monitoring, human accountability, and a way to appeal decisions.

## Quick self-check

1. What are the six inputs and the one target?
2. Why do we scale the inputs?
3. What does the hidden ReLU layer add?
4. Why is balanced accuracy useful alongside ordinary accuracy?
5. Why does a high test score not make this a real lending system?

## Finish line

You now have:

- A high-performing prediction using a financial Kaggle dataset
- Six understandable financial inputs and one approval probability
- A fully connected network with one eight-neuron ReLU layer
- A baseline, held-out test set, multiple metrics, and confusion matrix
- Editable applicant examples and an honest review of model mistakes